## HW Assignment 1

In this assignment, we will learn how to use Apache Spark RDDs and explore MapReduce and distributed processing. For each question, add your code below the question and run the code.

Note: Your code can be in more than one cell if you choose.

Let's start by running the code below to start up a local Spark instance.

In [40]:
import os

os.environ.pop("SPARK_HOME", None)
os.environ.pop("JAVA_HOME", None)

In [41]:
!pip install -q pyspark

from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("HW1").getOrCreate()
sc = spark.sparkContext
sc

<SparkContext master=local[*] appName=HW1>

In [42]:
from google.colab import drive
drive.mount('/content/gdrive')
FILE_PATH = '/content/gdrive/MyDrive/Colab Notebooks/HW1/alice.txt'

Drive already mounted at /content/gdrive; to attempt to forcibly remount, call drive.mount("/content/gdrive", force_remount=True).


1. Load the Alice in Wonderland text file into an RDD

In [43]:
alice_rdd = sc.textFile(FILE_PATH)
print(f"Total lines: {alice_rdd.count()}")
alice_rdd.take(3)

Total lines: 3773


['The Project Gutenberg EBook of Alice’s Adventures in Wonderland, by Lewis Carroll',
 '',
 'This eBook is for the use of anyone anywhere at no cost and with']

2. Transform all characters lowercase and remove all non-character symbols using the map function and store this in a new RDD.

In [44]:
import re

def clean_line(line):
  line = line.lower()
  line = re.sub(r'[^a-z ]', '', line)
  return line

cleaned_rdd = alice_rdd.map(clean_line)
cleaned_rdd.take(3)

['the project gutenberg ebook of alices adventures in wonderland by lewis carroll',
 '',
 'this ebook is for the use of anyone anywhere at no cost and with']

3. Write code to calculate the distribution of word length for the book Alice in Wonderland. Print the distribution in this notebook.

In [49]:
word_len_dist = (
    cleaned_rdd
    .flatMap(lambda line: line.split())
    .filter(lambda w: len(w) > 0)
    .map(lambda word: (len(word), 1))
    .reduceByKey(lambda a, b: a + b)
    .sortByKey()
)

print(f"{'Word Length':<15} {'Count'}")
print("-" * 25)
for length, count in word_len_dist.take(50):
    print(f"{length:<15} {count}")

Word Length     Count
-------------------------
1               1148
2               4763
3               7446
4               6090
5               3576
6               2241
7               1916
8               869
9               675
10              382
11              242
12              80
13              29
14              24
15              10
16              2
18              2
19              2
21              2
22              2
27              1
40              1


4. N-grams are contiguous sequences of n words. Write a function to take a string containing multiple words and n as input and return a list of n-grams (the length of the n-grams is a parameter inputted into the function).


Note: it's fine to only create ngrams out of each row of the text. For the purpose of this exercise, there is no need to combine rows of text.

In [46]:
def get_ngrams(text, n):
    words = text.split()
    if len(words) < n:
        return []
    return [tuple(words[i:i+n]) for i in range(len(words) - n + 1)]

print(get_ngrams("the quick brown fox", 2))

[('the', 'quick'), ('quick', 'brown'), ('brown', 'fox')]


5. Transform the RDD containing the book Alice in Wonderland to an RDD containing all 2-grams from each row in the book.

In [47]:
bigrams_rdd = cleaned_rdd.flatMap(lambda line: get_ngrams(line, 2))

print(f"Total 2-grams: {bigrams_rdd.count()}")
bigrams_rdd.take(5)

Total 2-grams: 26699


[('the', 'project'),
 ('project', 'gutenberg'),
 ('gutenberg', 'ebook'),
 ('ebook', 'of'),
 ('of', 'alices')]

6. Write code to find the distribution of all 2-grams in the book and print it below.

In [48]:
bigram_dist = (
    bigrams_rdd
    .map(lambda bigram: (bigram, 1))
    .reduceByKey(lambda a, b: a + b)
    .sortBy(lambda x: x[1], ascending=False)   # most frequent first
)

print(f"{'2-gram':<30} {'Count'}")
print("-" * 40)
for bigram, count in bigram_dist.take(20):
    print(f"{' '.join(bigram):<30} {count}")

2-gram                         Count
----------------------------------------
said the                       206
of the                         154
said alice                     112
in a                           101
in the                         90
to the                         84
and the                        77
it was                         71
the queen                      60
as she                         59
at the                         58
the king                       58
a little                       57
she had                        55
the mock                       54
she was                        53
to be                          52
with the                       52
and she                        50
the hatter                     50
